In [3]:
import sqlite3
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegressionCV
from imblearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

conn= sqlite3.connect('/home/shail/interview-prep/01_SQL_Warmups/oracle_hr.db')

## Question 1: SQL – Average Total Compensation by Department

### The Question

Write a SQL query against the standard Oracle HR `employees` table to find the `department_id` and the average total compensation (`salary + commission amount`) for departments where this average total compensation is **strictly greater than 10,000**.

Only include employees hired after **January 1, 2000**.

> Total compensation = `salary + (salary * commission_pct)`, where `commission_pct` can be `NULL`.

### Solution

In [5]:
pd.read_sql_query(sql= """
SELECT
    department_id,
    AVG(salary + (salary * COALESCE(commission_pct, 0))) AS avg_total_comp
FROM employees
WHERE hire_date > '2000-01-01'
GROUP BY department_id
HAVING AVG(salary + (salary * COALESCE(commission_pct, 0))) > 10000;
""", con= conn)

,department_id,avg_total_comp
0,80,11092.352941
1,90,19333.333333
2,110,10154.000000


# Question 2: Valid User Indices in O(N)

## The Question

Write a function `get_valid_user_indices(user_ids, banned_users)` that takes:

- a list of sequential transactions (`user_ids`)
- a list of banned users

Return a dictionary where:

- **keys** = valid user IDs
- **values** = list of the exact **0-indexed positions** where that user appeared in the original log.

The solution must run in **O(N)** time.

## Solution

In [7]:
#Data:
import random

# Generate mock data
random.seed(42)
user_ids = [random.randint(1, 10) for _ in range(15)]
banned_users = [2, 5, 8]

print("Transaction Log:", user_ids)
print("Banned Users:", banned_users)

Transaction Log: [2, 1, 5, 4, 4, 3, 2, 9, 2, 10, 7, 1, 1, 2, 4]
Banned Users: [2, 5, 8]


In [8]:
def get_valid_user_indices(user_ids: list, banned_users: list) -> dict:
    # O(1) lookups for banned users
    banned_set = set(banned_users)

    result = {}

    # Process the stream in a single pass (O(N))
    for index, user in enumerate(user_ids):
        if user not in banned_set:
            # Initialize the list if the key doesn't already exist
            result.setdefault(user, []).append(index)

    return result

In [9]:
get_valid_user_indices(user_ids= user_ids, banned_users= banned_users)

{1: [1, 11, 12], 4: [3, 4, 14], 3: [5], 9: [7], 10: [9], 7: [10]}

## Key Explanations

- **`set(banned_users)`**
  - Membership lookup in a list is **O(N)**.
  - Membership lookup in a set is **O(1)**.
  - Converting once at the beginning makes the overall algorithm linear.

- **Order Preservation**
  - Iterate directly over the original list using `enumerate()`.
  - Do **not** convert `user_ids` into a set.
  - A set removes duplicates and destroys chronological ordering.

---

# Question 3: Class Weights vs. SMOTE

## The Questions

1. What is the mechanical difference between using `class_weight='balanced'` versus generating synthetic data using **SMOTE**?
2. When would you prefer one?
3. If you use SMOTE, where in your cross-validation pipeline must you apply it, and why?

## Class Weights vs. SMOTE

### Mechanics

**`class_weight='balanced'`**

- Changes the model's **loss function**.
- Misclassifying minority-class samples incurs a larger penalty.
- The training data itself is **unchanged**.

**SMOTE**

- Changes the **training dataset**.
- Uses **K-Nearest Neighbors (KNN)** to interpolate entirely new synthetic minority-class samples.
- Expands the feature space with generated observations.

### When to Use Each

**Prefer `class_weight` when:**

- Working with high-dimensional data (e.g., NLP/text).
- Working with categorical features.
- Euclidean distance is not meaningful.

**Prefer SMOTE when:**

- Features are continuous numeric variables.
- Data has relatively low dimensionality.
- Interpolation between nearby points makes geometric sense.

### Why?

SMOTE depends on Euclidean distance.

In high-dimensional spaces, distances become less meaningful due to the **curse of dimensionality**, making synthetic samples less reliable.

---

## Data Leakage

SMOTE must be applied **strictly inside the cross-validation loop**, affecting **only the training folds**.

### Why?

If SMOTE is applied **before** splitting:

- KNN may generate a synthetic point using:
  - one sample that later belongs to the training set, and
  - another sample that later belongs to the validation/test set.

This leaks information from the validation data into training, producing overly optimistic evaluation metrics.

---

# Question 4: Logistic Regression Pipeline with SMOTE

## The Question

Write a pipeline using Scikit-learn and Imbalanced-learn to:

- Standard Scale the features
- Apply SMOTE
- Train Logistic Regression
- Evaluate using **5-fold Stratified Cross-Validation**
- Print the mean F1-score

## Solution

In [11]:
# Data:
from sklearn.datasets import make_classification
import pandas as pd

# Generate mock imbalanced dataset
X, y = make_classification(
    n_samples=5000, 
    n_features=10, 
    n_informative=3,
    n_redundant=1,
    weights=[0.95], # 95% majority class, 5% minority
    flip_y=0, 
    random_state=42
)

In [12]:

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

# 1. Define the pipeline using imblearn
pipeline = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression(random_state=42))
])

# 2. Define the cross-validation strategy
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# 3. Evaluate the pipeline
scores = cross_val_score(
    estimator=pipeline,
    X=X,
    y=y,
    cv=cv,
    scoring='f1'
)

# 4. Print the average F1-score
print(f"Mean F1-Score: {scores.mean():.4f}")

Mean F1-Score: 0.4771


## Key Explanations

### `imblearn.pipeline.Pipeline`

Use the pipeline from **Imbalanced-learn**, **not** Scikit-learn.

Why?

- Scikit-learn pipelines expect intermediate steps to implement `.transform()`.
- SMOTE performs **resampling**, not transformation.
- Imbalanced-learn's pipeline correctly performs resampling only on the training folds while leaving validation folds untouched.

---

### `cross_val_score`

Wrapping the entire pipeline inside `cross_val_score()` ensures that, for every fold:

1. Data is split.
2. The scaler is fit only on the training data.
3. SMOTE is applied only to the training data.
4. The classifier is trained.
5. Evaluation occurs on untouched validation data.

This prevents data leakage.

---

### `StratifiedKFold`

For imbalanced datasets (e.g., **95% / 5%**):

- Every fold preserves the original class ratio.
- Ensures each fold contains minority-class examples.

Using a standard `KFold` can produce validation folds containing few or even zero minority samples, leading to unreliable performance estimates.